In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
# from lib_file import lib_path
from urllib.parse import urlparse
from tld import get_tld
import os.path
import re
import pickle

In [2]:
with open(file="models/RandomForestClassifier_model.pkl", mode="rb") as file:
    model = pickle.load(file=file)

In [3]:
class_labels = {0:'Not WebPhishing',1:'WebPhishing'}

---

In [4]:
#First Directory Length
def fd_length(url):
    urlpath= urlparse(url).path
    try:
        return len(urlpath.split('/')[1])
    except:
        return 0

In [5]:
def tld_length(tld):
    try:
        return len(tld)
    except:
        return -1

In [6]:
def digit_count(url):
    digits = 0
    for i in url:
        if i.isnumeric():
            digits = digits + 1
    return digits

In [7]:
def letter_count(url):
    letters = 0
    for i in url:
        if i.isalpha():
            letters = letters + 1
    return letters

In [8]:
def no_of_dir(url):
    urldir = urlparse(url).path
    return urldir.count('/')

In [9]:
#Use of IP or not in domain
def having_ip_address(url):
    match = re.search(
        '(([01]?\\d\\d?|2[0-4]\\d|25[0-5])\\.([01]?\\d\\d?|2[0-4]\\d|25[0-5])\\.([01]?\\d\\d?|2[0-4]\\d|25[0-5])\\.'
        '([01]?\\d\\d?|2[0-4]\\d|25[0-5])\\/)|'  # IPv4
        '((0x[0-9a-fA-F]{1,2})\\.(0x[0-9a-fA-F]{1,2})\\.(0x[0-9a-fA-F]{1,2})\\.(0x[0-9a-fA-F]{1,2})\\/)' # IPv4 in hexadecimal
        '(?:[a-fA-F0-9]{1,4}:){7}[a-fA-F0-9]{1,4}', url)  # Ipv6
    if match:
        # print match.group()
        return -1
    else:
        # print 'No matching pattern found'
        return 1

In [10]:
def shortening_service(url):
    match = re.search('bit\.ly|goo\.gl|shorte\.st|go2l\.ink|x\.co|ow\.ly|t\.co|tinyurl|tr\.im|is\.gd|cli\.gs|'
                      'yfrog\.com|migre\.me|ff\.im|tiny\.cc|url4\.eu|twit\.ac|su\.pr|twurl\.nl|snipurl\.com|'
                      'short\.to|BudURL\.com|ping\.fm|post\.ly|Just\.as|bkite\.com|snipr\.com|fic\.kr|loopt\.us|'
                      'doiop\.com|short\.ie|kl\.am|wp\.me|rubyurl\.com|om\.ly|to\.ly|bit\.do|t\.co|lnkd\.in|'
                      'db\.tt|qr\.ae|adf\.ly|goo\.gl|bitly\.com|cur\.lv|tinyurl\.com|ow\.ly|bit\.ly|ity\.im|'
                      'q\.gs|is\.gd|po\.st|bc\.vc|twitthis\.com|u\.to|j\.mp|buzurl\.com|cutt\.us|u\.bb|yourls\.org|'
                      'x\.co|prettylinkpro\.com|scrnch\.me|filoops\.info|vzturl\.com|qr\.net|1url\.com|tweez\.me|v\.gd|'
                      'tr\.im|link\.zip\.net',
                      url)
    if match:
        return -1
    else:
        return 1

---

In [11]:
input_url = "https://www.zimbio.com/Lynda+Bellingham/pictures/pro"

In [12]:
urldata = pd.DataFrame()
urldata['url'] = [input_url]
urldata.head()

,url
0,https://www.zimbio.com/Lynda+Bellingham/pictur...


In [13]:
urldata['url_length'] = urldata['url'].apply(lambda i: len(str(i)))
urldata['hostname_length'] = urldata['url'].apply(lambda i: len(urlparse(i).netloc))
urldata['path_length'] = urldata['url'].apply(lambda i: len(urlparse(i).path))
urldata['fd_length'] = urldata['url'].apply(lambda i: fd_length(i))
urldata['tld'] = urldata['url'].apply(lambda i: get_tld(i,fail_silently=True))
urldata['tld_length'] = urldata['tld'].apply(lambda i: tld_length(i))
urldata = urldata.drop(labels="tld",axis=1)
urldata['count-'] = urldata['url'].apply(lambda i: i.count('-'))
urldata['count@'] = urldata['url'].apply(lambda i: i.count('@'))
urldata['count?'] = urldata['url'].apply(lambda i: i.count('?'))
urldata['count%'] = urldata['url'].apply(lambda i: i.count('%'))
urldata['count.'] = urldata['url'].apply(lambda i: i.count('.'))
urldata['count='] = urldata['url'].apply(lambda i: i.count('='))
urldata['count-http'] = urldata['url'].apply(lambda i : i.count('http'))
urldata['count-https'] = urldata['url'].apply(lambda i : i.count('https'))
urldata['count-www'] = urldata['url'].apply(lambda i: i.count('www'))
urldata['count-digits']= urldata['url'].apply(lambda i: digit_count(i))
urldata['count-letters']= urldata['url'].apply(lambda i: letter_count(i))
urldata['count_dir'] = urldata['url'].apply(lambda i: no_of_dir(i))
urldata['use_of_ip'] = urldata['url'].apply(lambda i: having_ip_address(i))
urldata['short_url'] = urldata['url'].apply(lambda i: shortening_service(i))
urldata = urldata.iloc[:,1:]

In [14]:
print(urldata.values)

[[52 14 30 16  3  0  0  0  0  2  0  1  1  1  0 43  3  1  1]]


In [15]:
model_pred = model.predict_proba(urldata.values)
model_pred

array([[9.99520575e-01, 4.79424690e-04]])

In [16]:
model_pred.argmax()

0

In [17]:
model_pred = model.predict_proba(urldata.values)
class_index = model_pred.argmax()
class_label =class_labels[class_index]
probability = model_pred[0][class_index]

print(f"CLASS INDEX : {class_index}")
print(f"CLASS LABEL : {class_label}")
print(f"PROBABILITY : {probability*100.0}")

CLASS INDEX : 0
CLASS LABEL : Not WebPhishing
PROBABILITY : 99.95205753096285
